# A simple categorical Naive Bayes classifier

Build a small classifier from scratch to exemplify the **scikit-learn estimator API**, then compare it with scikit-learn's categorical Naive Bayes on a held-out muffin dataset.

## Contents

1. [Data: The muffin dataset](#load-the-muffin-dataset)
2. [The Naive Bayes classifier](#the-naive-bayes-classifier)
3. [Estimating probabilities](#estimating-probabilities)
4. [Fit the classifier](#fit-the-classifier)
5. [Inspect what was learned](#inspect-what-was-learned)
6. [Predict](#predict)
7. [Predict probabilities](#predict-probabilities)
8. [Discussion](#discussion)
9. [Muffin train-test comparison](#muffin-train-test-comparison)
10. [Takeaways](#takeaways)

All features are categorical. The from-scratch estimator uses pandas DataFrames so it can count and look up values by feature name; this keeps features aligned even when input columns are in a different order.

## Load the Muffin dataset

In [ ]:
## Data: The Muffin dataset

from pathlib import Path
import pandas as pd

data_dir = Path("../../data/muffin")
muffin_train = data_dir / "muffin_train.csv"
muffin_test = data_dir / "muffin_test.csv"

train = pd.read_csv(muffin_train, index_col="ID")
test = pd.read_csv(muffin_test, index_col="ID")

print("Training set:", train.shape, "head: \n", train.head())
print("Test set:", test.shape, "head: \n", test.head())

X_train = train.drop(columns="Sold quickly")
y_train = train["Sold quickly"]

X_test = test.drop(columns="Sold quickly")
y_test = test["Sold quickly"]

Training set: (16, 5) head: 
    Weekend       Time      Flavor   Size Sold quickly
ID                                                   
1       no    morning   chocolate  small          YES
2      yes  afternoon     vanilla  large           NO
3       no    morning  strawberry  small           NO
4      yes    morning   chocolate  large          YES
5       no      lunch     vanilla  small           NO
Test set: (8, 5) head: 
    Weekend       Time      Flavor   Size Sold quickly
ID                                                   
T1     yes    morning   chocolate  small          YES
T2      no  afternoon  strawberry  small           NO
T3      no    morning     vanilla  large           NO
T4     yes      lunch  strawberry  small           NO
T5      no      lunch   chocolate  small          YES


## The Naive Bayes Classifier

For a new example $\mathbf{x}=(x_1,\ldots,x_m)$, Naive Bayes predicts

$$
\hat{y}
=
\arg\max_c
P(Y=c)
\prod_{j=1}^{m}
P(X_j=x_j \mid Y=c).
$$

where:

- $c$ is a possible class,
- $P(Y=c)$ is the prior probability of class $c$,
- $x_j$ is the observed value of feature $X_j$,
- $P(X_j=x_j \mid Y=c)$ is the conditional probability of that feature value given class $c$.


**Think.** What must happen during `fit` ?

<details>
<summary><strong>Answer</strong></summary>

During `fit`, we count:

- how often each class occurs;
- how often each feature value occurs within each class;
- estimate and store probabilities thereof.

</details>

### Estimating probabilities

**Think.** How do we estimate probabilities?

<details>
<summary><strong>Answer</strong></summary>

From counts, we can use:

- **relative frequency**: simple, but susceptible to zero probabilities and unstable;
- **Laplace estimate**: adds pseudo-counts to avoid zero probabilities.

In our Naive Bayes implementation, class priors use relative frequencies and conditional feature probabilities use the Laplace estimate.

</details>

In [2]:
def estimate_probabilities(counts_dict, laplace=True):
    """Return probabilities from counts.

    If laplace=True, return Laplace-smoothed probabilities.
    Otherwise, return relative frequencies.
    """

    counts = counts_dict.copy()  # do not modify the original counts

    if laplace:
        # Add one pseudo-count to each possible value
        for key in counts:
            counts[key] += 1

    total = sum(counts.values())

    probabilities = {}
    for key, value in counts.items():
        probabilities[key] = value / total

    return probabilities



Test and compare probability estimates: with and without laplacean smoothing.

In [3]:
estimate_probabilities({'afternoon': 4, 'lunch': 3, 'morning': 0})

{'afternoon': 0.5, 'lunch': 0.4, 'morning': 0.1}

In [4]:
estimate_probabilities({'afternoon': 4, 'lunch': 3, 'morning': 0}, laplace= False)

{'afternoon': 0.5714285714285714, 'lunch': 0.42857142857142855, 'morning': 0.0}

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin


class CategoricalNaiveBayes(ClassifierMixin, BaseEstimator):
    """Naive Bayes classifier for categorical pandas DataFrames."""

    def fit(self, X, y):
        """Learn class and conditional feature-value probabilities."""

        self.classes_ = np.unique(y)
        self.feature_names_ = list(X.columns)

        # Counts
        self.counts_ = {}

        # P(class): relative frequencies
        self.counts_["class"] = y.value_counts().to_dict()
        self.class_probabilities_ = estimate_probabilities(self.counts_["class"], laplace=False)

        # P(feature=value | class): Laplace estimates
        self.probabilities_ = {}
        for class_label in self.classes_:
            self.counts_[class_label] = {}
            self.probabilities_[class_label] = {}
            for feature_name in self.feature_names_:
                counts = {}
                # Include all possible feature values, including those with count 0 in this class
                for value in X[feature_name].unique():
                    count = np.sum((y == class_label) & (X[feature_name] == value))
                    counts[value] = count

                self.counts_[class_label][feature_name] = counts
                self.probabilities_[class_label][feature_name] = (estimate_probabilities(counts))

        return self

    def predict_proba(self, X):
        """Return class probabilities for each example in X."""

        probabilities = []

        for _, row in X.iterrows():
            scores = []

            for class_label in self.classes_:

                # P(class)
                score = self.class_probabilities_[class_label]

                # Multiply by P(feature=value | class)
                for feature_name in self.feature_names_:
                    value = row[feature_name]
                    score *= self.probabilities_[class_label][feature_name].get(value, 0)

                scores.append(score)

            scores = np.array(scores)
            probabilities.append(scores / scores.sum())

        return np.array(probabilities)

    def predict(self, X):
        """Return the most probable class for each example in X."""
        probabilities = self.predict_proba(X)
        best = np.argmax(probabilities, axis=1)

        return self.classes_[best]

## Fit the classifier

Notice that `fit` does not return predictions. It only learns from the training data and stores what it learned on `self`.

In [6]:
clf = CategoricalNaiveBayes()
clf.fit(X_train, y_train)


CategoricalNaiveBayes()

## Inspect what was learned

In [7]:
clf.classes_

array(['NO', 'YES'], dtype=object)

In [8]:
clf.class_probabilities_ 


{'YES': 0.5, 'NO': 0.5}

In [9]:
clf.counts_



{'class': {'YES': 8, 'NO': 8},
 'NO': {'Weekend': {'no': 4, 'yes': 4},
  'Time': {'morning': 1, 'afternoon': 4, 'lunch': 3},
  'Flavor': {'chocolate': 0, 'vanilla': 4, 'strawberry': 4},
  'Size': {'small': 4, 'large': 4}},
 'YES': {'Weekend': {'no': 4, 'yes': 4},
  'Time': {'morning': 4, 'afternoon': 2, 'lunch': 2},
  'Flavor': {'chocolate': 6, 'vanilla': 2, 'strawberry': 0},
  'Size': {'small': 5, 'large': 3}}}

In [10]:
clf.probabilities_ 

{'NO': {'Weekend': {'no': 0.5, 'yes': 0.5},
  'Time': {'morning': 0.18181818181818182,
   'afternoon': 0.45454545454545453,
   'lunch': 0.36363636363636365},
  'Flavor': {'chocolate': 0.09090909090909091,
   'vanilla': 0.45454545454545453,
   'strawberry': 0.45454545454545453},
  'Size': {'small': 0.5, 'large': 0.5}},
 'YES': {'Weekend': {'no': 0.5, 'yes': 0.5},
  'Time': {'morning': 0.45454545454545453,
   'afternoon': 0.2727272727272727,
   'lunch': 0.2727272727272727},
  'Flavor': {'chocolate': 0.6363636363636364,
   'vanilla': 0.2727272727272727,
   'strawberry': 0.09090909090909091},
  'Size': {'small': 0.6, 'large': 0.4}}}

## Predict

`predict` returns one class label per row.

In [11]:
y_hat = clf.predict(X_test)
y_hat

array(['YES', 'NO', 'YES', 'NO', 'YES', 'NO', 'NO', 'YES'], dtype=object)

## Predict probabilities

`predict_proba` returns one row per example and one column per class.

Column `j` corresponds to `clf.classes_[j]`.

In [12]:
clf.classes_

array(['NO', 'YES'], dtype=object)

In [13]:
y_hat_proba = clf.predict_proba(X_test)
y_hat_proba

array([[0.04545455, 0.95454545],
       [0.87412587, 0.12587413],
       [0.45454545, 0.54545455],
       [0.84745763, 0.15254237],
       [0.1369863 , 0.8630137 ],
       [0.69832402, 0.30167598],
       [0.71428571, 0.28571429],
       [0.2293578 , 0.7706422 ]])

In [14]:
results = X_test.copy()

results["y"] = y_test
results["y_hat"] = y_hat

for i, class_label in enumerate(clf.classes_):
    results[f"P({class_label})"] = y_hat_proba[:, i]

results

,Weekend,Time,Flavor,Size,y,y_hat,P(NO),P(YES)
ID,,,,,,,,
T1,yes,morning,chocolate,small,YES,YES,0.045455,0.954545
T2,no,afternoon,strawberry,small,NO,NO,0.874126,0.125874
T3,no,morning,vanilla,large,NO,YES,0.454545,0.545455
T4,yes,lunch,strawberry,small,NO,NO,0.847458,0.152542
T5,no,lunch,chocolate,small,YES,YES,0.136986,0.863014
T6,yes,afternoon,vanilla,small,NO,NO,0.698324,0.301676
T7,yes,morning,strawberry,large,YES,NO,0.714286,0.285714
T8,no,afternoon,chocolate,large,YES,YES,0.229358,0.770642


__________________________________________________________

## Muffin data: train-test comparison

We now use the muffin training dataset to compare our CategoricalNaiveBayes classifier with scikit-learn's `CategoricalNB` estimator. 

We use pipelines: `OrdinalEncoder` converts the categorical features to non-negative integer codes for scikit-learn. It is fitted on the training data, then used to transform the test data.

In [15]:
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline


# From-scratch implementation
scratch_clf = CategoricalNaiveBayes()
scratch_clf.fit(X_train, y_train)
scratch_predictions = scratch_clf.predict(X_test)


# scikit-learn implementation
sklearn_clf = Pipeline([
    ("encoder", OrdinalEncoder(dtype=int)),
    ("classifier", CategoricalNB(alpha=1.0))  # Laplace smoothing
])

sklearn_clf.fit(X_train, y_train)
sklearn_predictions = sklearn_clf.predict(X_test)


# Compare predictions
predictions = pd.DataFrame({
    "Actual": y_test,
    "From scratch": scratch_predictions,
    "scikit-learn": sklearn_predictions,
})

display(predictions)


# Compare accuracy
comparison = pd.DataFrame({
    "Model": ["From scratch", "scikit-learn CategoricalNB"],
    "Test accuracy": [
        accuracy_score(y_test, scratch_predictions),
        accuracy_score(y_test, sklearn_predictions),
    ],
})

comparison

,Actual,From scratch,scikit-learn
ID,,,
T1,YES,YES,YES
T2,NO,NO,NO
T3,NO,YES,YES
T4,NO,NO,NO
T5,YES,YES,YES
T6,NO,NO,NO
T7,YES,NO,NO
T8,YES,YES,YES


,Model,Test accuracy
0,From scratch,0.75
1,scikit-learn CategoricalNB,0.75


## Notes

1. Our implementation is deliberately symbolic and uses feature names from a pandas DataFrame. Scikit-learn estimators generally accept array-like inputs, including NumPy arrays and, in many cases, pandas DataFrames.

2. What is the role of

```python
from sklearn.base import BaseEstimator, ClassifierMixin

class CategoricalNaiveBayes(ClassifierMixin, BaseEstimator):
    ...
```

It provides standard estimator behavior and compatibility with utilities such as `cross_val_score`, `Pipeline`, and `GridSearchCV`. For the simple use case in this notebook, the simpler

```python
class CategoricalNaiveBayes:
    ...
```

would suffice.

## Takeaways

- `fit` learns information from the training set and stores it on `self`;
- `predict` and `predict_proba` reuse that stored information;
- `self.classes_` defines the class order used by `predict_proba`;
- conditional feature probabilities are estimated with Laplace smoothing;
- the same `fit`, `predict`, and `predict_proba` estimator interface can hide different implementations;
- scikit-learn's `CategoricalNB` expects categorical values encoded as non-negative integers;
- a `Pipeline` keeps preprocessing and the classifier together and fits preprocessing only on the training data.